In [1]:
!pip install -q langgraph langchain-groq langchain-core pydantic

In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [3]:
from typing import TypedDict, Optional, List

class AgentState(TypedDict):
    query: str
    intent: Optional[str]
    confidence: Optional[float]
    sentiment: Optional[str]
    response: Optional[str]
    escalated: bool
    conversation_history: List[str]

In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [5]:
from pydantic import BaseModel, Field
from typing import Literal

class ClassificationResult(BaseModel):
    intent: Literal["billing", "technical", "faq", "escalation"] = Field(
        description="Query ka category"
    )
    confidence: float = Field(description="0 to 1 ke beech confidence score")
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="User ka tone/mood"
    )
    reasoning: str = Field(description="Ek line mein reason kyun ye category chuni")

In [6]:
structured_llm = llm.with_structured_output(ClassificationResult)

def classifier_node(state: AgentState) -> AgentState:
    result = structured_llm.invoke(
        f"""Classify this customer support query.

Query: {state['query']}

Rules:
- If user mentions "manager", "refund now", "legal", "cancel subscription" → escalation
- billing = payments, invoices, subscriptions, refunds
- technical = bugs, errors, how-to, troubleshooting
- faq = general info questions
"""
    )

    state["intent"] = result.intent
    state["confidence"] = result.confidence
    state["sentiment"] = result.sentiment

    return state

In [7]:
test_state = {"query": "My refund is not processed yet, I want to talk to manager", "escalated": False, "conversation_history": []}
print(classifier_node(test_state))

{'query': 'My refund is not processed yet, I want to talk to manager', 'escalated': False, 'conversation_history': [], 'intent': 'escalation', 'confidence': 0.9, 'sentiment': 'negative'}


In [8]:
def billing_node(state: AgentState) -> AgentState:
    response = llm.invoke(
        f"""You are a Billing Support Agent. Answer this customer query professionally.

Query: {state['query']}

Handle topics like payments, refunds, invoices, subscriptions."""
    )
    state["response"] = response.content
    return state


def technical_node(state: AgentState) -> AgentState:
    response = llm.invoke(
        f"""You are a Technical Support Agent. Help troubleshoot this issue.

Query: {state['query']}

Give clear step-by-step troubleshooting if needed."""
    )
    state["response"] = response.content
    return state


def faq_node(state: AgentState) -> AgentState:
    response = llm.invoke(
        f"""You are a General FAQ Agent. Answer this general question clearly and briefly.

Query: {state['query']}"""
    )
    state["response"] = response.content
    return state

In [9]:
def escalation_node(state: AgentState) -> AgentState:
    state["response"] = (
        " Your query has been escalated to a human agent. "
        "A support representative will contact you shortly. "
        f"(Reason: {state.get('sentiment', 'unknown')} sentiment / low confidence detected)"
    )
    state["escalated"] = True
    return state

In [27]:
def route_query(state: AgentState) -> str:
    if state["confidence"] <= 0.6:
        return "escalation"
    return state["intent"]

In [28]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

# Nodes add karo
graph.add_node("classifier", classifier_node)
graph.add_node("billing", billing_node)
graph.add_node("technical", technical_node)
graph.add_node("faq", faq_node)
graph.add_node("escalation", escalation_node)

# Entry point
graph.set_entry_point("classifier")

# Conditional edge — classifier ke baad route_query decide karega kahan jana hai
graph.add_conditional_edges(
    "classifier",
    route_query,
    {
        "billing": "billing",
        "technical": "technical",
        "faq": "faq",
        "escalation": "escalation"
    }
)

# Sab specialist nodes ke baad END
graph.add_edge("billing", END)
graph.add_edge("technical", END)
graph.add_edge("faq", END)
graph.add_edge("escalation", END)

# Compile
app = graph.compile()

In [29]:
result = app.invoke({
    "query": "How do I reset my password?",
    "escalated": False,
    "conversation_history": []
})

print("Intent:", result["intent"])
print("Escalated:", result["escalated"])
print("Response:", result["response"])

Intent: technical
Escalated: False
Response: Resetting your password is a straightforward process. Here are the steps to follow:

**Method 1: Reset Password using the Forgot Password Option**

1. Go to the login page of the website or application you're trying to access.
2. Click on the "Forgot Password" or "Reset Password" link, usually located below the login form.
3. Enter your username or email address associated with your account.
4. Click on the "Reset Password" or "Send Reset Link" button.
5. Check your email inbox for a password reset email from the website or application.
6. Open the email and click on the password reset link.
7. Enter a new password and confirm it by re-entering it in the required field.
8. Click on the "Reset Password" or "Save Changes" button to save your new password.

**Method 2: Reset Password using Account Settings**

1. Log in to your account using your current password (if you remember it).
2. Go to your account settings or profile page.
3. Look for t

In [30]:
result2 = app.invoke({
    "query": "This is the third time I'm asking! I want a refund NOW or I'm taking legal action!",
    "escalated": False,
    "conversation_history": []
})

print("Intent:", result2["intent"])
print("Sentiment:", result2["sentiment"])
print("Escalated:", result2["escalated"])
print("Response:", result2["response"])

Intent: escalation
Sentiment: negative
Escalated: True
Response:  Your query has been escalated to a human agent. A support representative will contact you shortly. (Reason: negative sentiment / low confidence detected)


In [31]:
result3 = app.invoke({
    "query": "hmm okay thanks I guess",
    "escalated": False,
    "conversation_history": []
})

print("Intent:", result3["intent"])
print("Confidence:", result3["confidence"])
print("Escalated:", result3["escalated"])

Intent: faq
Confidence: 0.4
Escalated: True


In [32]:
!pip install -q streamlit pyngrok

In [33]:
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

In [34]:
%%writefile app.py

import streamlit as st
import os
from typing import TypedDict, Optional, List, Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

# ---------- Config ----------
os.environ["GROQ_API_KEY"] = st.secrets["GROQ_API_KEY"]

st.set_page_config(page_title="Customer Support AI")

# ---------- State ----------
class AgentState(TypedDict):
    query: str
    intent: Optional[str]
    confidence: Optional[float]
    sentiment: Optional[str]
    response: Optional[str]
    escalated: bool
    conversation_history: List[str]

class ClassificationResult(BaseModel):
    intent: Literal["billing", "technical", "faq", "escalation"] = Field(description="Query category")
    confidence: float = Field(description="0 to 1 confidence score")
    sentiment: Literal["positive", "neutral", "negative"] = Field(description="User's tone")
    reasoning: str = Field(description="One line reason")

# ---------- LLM ----------
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
structured_llm = llm.with_structured_output(ClassificationResult)

# ---------- Nodes ----------
def classifier_node(state: AgentState) -> AgentState:
    result = structured_llm.invoke(f"""Classify this customer support query.
Query: {state['query']}
Rules:
- If user mentions "manager", "refund now", "legal", "cancel subscription" -> escalation
- billing = payments, invoices, subscriptions, refunds
- technical = bugs, errors, how-to, troubleshooting
- faq = general info questions""")
    state["intent"] = result.intent
    state["confidence"] = result.confidence
    state["sentiment"] = result.sentiment
    return state

def billing_node(state: AgentState) -> AgentState:
    response = llm.invoke(f"You are a Billing Support Agent. Answer professionally.\nQuery: {state['query']}")
    state["response"] = response.content
    return state

def technical_node(state: AgentState) -> AgentState:
    response = llm.invoke(f"You are a Technical Support Agent. Give step-by-step help.\nQuery: {state['query']}")
    state["response"] = response.content
    return state

def faq_node(state: AgentState) -> AgentState:
    response = llm.invoke(f"You are a General FAQ Agent. Answer clearly and briefly.\nQuery: {state['query']}")
    state["response"] = response.content
    return state

def escalation_node(state: AgentState) -> AgentState:
    state["response"] = (
        "Your query has been escalated to a human agent. "
        "A support representative will contact you shortly. "
        f"(Reason: {state.get('sentiment','unknown')} sentiment / low confidence detected)"
    )
    state["escalated"] = True
    return state

def route_query(state: AgentState) -> str:
    if state["confidence"] <= 0.6:
        return "escalation"
    return state["intent"]

# ---------- Graph ----------
@st.cache_resource
def build_graph():
    graph = StateGraph(AgentState)
    graph.add_node("classifier", classifier_node)
    graph.add_node("billing", billing_node)
    graph.add_node("technical", technical_node)
    graph.add_node("faq", faq_node)
    graph.add_node("escalation", escalation_node)
    graph.set_entry_point("classifier")
    graph.add_conditional_edges("classifier", route_query, {
        "billing": "billing", "technical": "technical",
        "faq": "faq", "escalation": "escalation"
    })
    graph.add_edge("billing", END)
    graph.add_edge("technical", END)
    graph.add_edge("faq", END)
    graph.add_edge("escalation", END)
    return graph.compile()

app_graph = build_graph()

# ---------- UI ----------
st.title(" Customer Support Multi-Agent System")
st.caption("LangGraph | Conditional Routing + Escalation")

if "history" not in st.session_state:
    st.session_state.history = []

query = st.text_area("Enter your query:", height=100)

if st.button("Submit") and query.strip():
    with st.spinner("Processing..."):
        result = app_graph.invoke({
            "query": query,
            "escalated": False,
            "conversation_history": []
        })
    st.session_state.history.append(result)

for r in reversed(st.session_state.history):
    st.markdown("---")
    st.markdown(f"**Query:** {r['query']}")
    col1, col2, col3 = st.columns(3)
    col1.metric("Intent", r["intent"])
    col2.metric("Confidence", f"{r['confidence']:.2f}")
    col3.metric("Sentiment", r["sentiment"])

    if r["escalated"]:
        st.error(" Escalated to Human Agent")
    st.info(r["response"])

Overwriting app.py


In [35]:
import os
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/secrets.toml", "w") as f:
    f.write(f'GROQ_API_KEY = "{userdata.get("GROQ_API_KEY")}"\n')

In [36]:
!pkill -f streamlit
!pkill -f ngrok

In [37]:
import time
time.sleep(3)

from pyngrok import ngrok
ngrok.kill()

public_url = ngrok.connect(8501)
print("APP URL:", public_url)

import subprocess
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(6)

APP URL: NgrokTunnel: "https://postnasal-angled-sessions.ngrok-free.dev" -> "http://localhost:8501"
